# Uber NYC Hot-Zones — Data Exploration

## Context
Uber drivers often lack visibility on high-demand areas. This project analyzes NYC pickup data 
(April–September 2014 and January–June 2015) to identify hot-zones that could guide driver positioning.

## Scope
- **2014 data:** 6 months of GPS-based pickup records (~4.5M rows)
- **2015 data:** 6 months of zone-based pickup records (~14M rows)  
- **Geographic reference:** NYC TLC taxi zone shapefile (263 zones)

## Notebook objectives
1. Load and preprocess all available data
2. Bridge 2014 GPS coordinates to taxi zones via spatial join
3. Explore temporal and geographic patterns
4. Export preprocessed data for clustering (NB03) and analysis (NB04)

In [1]:
import sys
import warnings

sys.path.insert(0, "src")
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data import load_2014_data, load_2015_data, load_zone_lookup, load_zone_shapefile
from src.preprocessing import (
    parse_timestamps_2014,
    parse_timestamps_2015,
    filter_nyc_bounds,
    normalize_coordinates,
    map_2015_zones,
)
from src.zones import spatial_join_2014, compute_zone_centroids
from src.config import OUTPUT_DIR, DAY_NAMES

print("All imports loaded successfully.")

All imports loaded successfully.


## 1. Data Loading

We use all available data from the NYC TLC Uber dataset:
- **2014 (Apr–Sep):** 6 CSV files with GPS coordinates (`Lat`, `Lon`) and dispatch `Base`
- **2015 (Jan–Jun):** 1 CSV with `locationID` referencing NYC taxi zones (no GPS coordinates)
- **Taxi zone shapefile:** 263 polygon zones for spatial mapping

**Why all months?** The assignment provides the complete dataset. Using all available data produces 
more robust temporal patterns and allows year-over-year comparison.

In [2]:
df_2014 = load_2014_data()
print(f"\nShape: {df_2014.shape}")
print(f"Memory: {df_2014.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df_2014.head()


Shape: (4534327, 4)
Memory: 342.2 MB


,Date/Time,Lat,Lon,Base
0,4/1/2014 0:11:00,40.769001,-73.954903,B02512
1,4/1/2014 0:17:00,40.726700,-74.034500,B02512
2,4/1/2014 0:21:00,40.731602,-73.987297,B02512
3,4/1/2014 0:28:00,40.758801,-73.977600,B02512
4,4/1/2014 0:33:00,40.759399,-73.972198,B02512


In [3]:
df_2015 = load_2015_data()
print(f"\nShape: {df_2015.shape}")
df_2015.head()


Shape: (14270479, 4)


,Dispatching_base_num,Pickup_date,Affiliated_base_num,locationID
0,B02617,2015-05-17 09:47:00,B02617,141
1,B02617,2015-05-17 09:47:00,B02617,65
2,B02617,2015-05-17 09:47:00,B02617,100
3,B02617,2015-05-17 09:47:00,B02774,80
4,B02617,2015-05-17 09:47:00,B02617,90


In [4]:
zone_lookup = load_zone_lookup()
zones_gdf = load_zone_shapefile()
print(f"Zone lookup: {zone_lookup.shape[0]} zones")
print(f"Shapefile: {zones_gdf.shape[0]} polygons")
zone_lookup.head()

Zone lookup: 265 zones
Shapefile: 263 polygons


,LocationID,Borough,Zone
0,1,EWR,Newark Airport
1,2,Queens,Jamaica Bay
2,3,Bronx,Allerton/Pelham Gardens
3,4,Manhattan,Alphabet City
4,5,Staten Island,Arden Heights


## 2. Preprocessing

### 2.1 Timestamp Parsing
Both datasets require datetime parsing to extract temporal features (hour, day of week, month).
- **2014 format:** `m/d/Y H:M:S` — parsed with explicit format for performance
- **2015 format:** `Y-m-d H:M:S` — parsed with explicit format (14M rows, inference would be slow)

In [5]:
df_2014 = parse_timestamps_2014(df_2014)
print(f"2014 date range: {df_2014['datetime'].min()} to {df_2014['datetime'].max()}")
print(f"Months covered: {sorted(df_2014['month'].unique())}")

df_2015 = parse_timestamps_2015(df_2015)
print(f"\n2015 date range: {df_2015['datetime'].min()} to {df_2015['datetime'].max()}")
print(f"Months covered: {sorted(df_2015['month'].unique())}")

2014 date range: 2014-04-01 00:00:00 to 2014-09-30 22:59:00
Months covered: [np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9)]

2015 date range: 2015-01-01 00:00:05 to 2015-06-30 23:59:00
Months covered: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6)]


### 2.2 Geographic Filtering (2014 only)

The 2014 GPS data contains outliers outside NYC boundaries. We apply a bounding box filter:
- Latitude: 40.50°N – 40.92°N
- Longitude: -74.30 to -73.70

These bounds cover all five boroughs while excluding clear GPS errors.
The 2015 data uses pre-defined taxi zone IDs, so no geographic filtering is needed.

In [6]:
df_2014 = filter_nyc_bounds(df_2014)

### 2.3 Zone Mapping

To compare 2014 and 2015 data at zone level, we need a common reference frame:
- **2014:** Spatial join — map each GPS point to its containing taxi zone polygon
- **2015:** Direct lookup — merge `locationID` with the zone reference table

This spatial join uses the official NYC TLC taxi zone shapefile (263 polygons).

In [7]:
df_2014 = spatial_join_2014(df_2014, zones_gdf)

The TLC lookup table assigns separate LocationIDs to sub-polygons that share the same zone in the shapefile (e.g., 57 duplicates 56 for Corona, 104-105 duplicate 103 for Governor's Island). `map_2015_zones` remaps these before merging, so trip counts aggregate to the canonical zone ID and match the 2014 spatial join output.

In [8]:
df_2015 = map_2015_zones(df_2015, zone_lookup)

The zone mapping step also drops TLC zones 264 and 265 ("Unknown"), which represent pickups that could not be geocoded to any real taxi zone. These account for 0.044% of 2015 data (6,264 trips) and are standard data-quality exclusions in TLC taxi datasets. They are removed here rather than downstream so that all subsequent analyses operate on clean, spatially-resolved records only.

### 2.4 Coordinate Normalization (2014, for clustering)

KMeans clustering requires normalized features. Raw GPS coordinates are problematic:
- 1° latitude ≈ 111 km (constant)  
- 1° longitude ≈ 111 × cos(40.7°) ≈ 84.3 km at NYC's latitude

Without correction, min-max scaling on raw `[Lat, Lon]` would overweight longitude by ~24%, 
because the longitude range covers fewer kilometers than the latitude range for the same degree span.

To fix this, **longitude is first multiplied by cos(40.7128°) ≈ 0.759** before min-max scaling. 
This projects longitude into approximate kilometer-equivalent units, so that the scaler operates 
on `[Lat, Lon × cos(40.7128°)]` — both axes now represent roughly equal geographic distances per unit.

This cosine projection is sufficient for a bounded urban area like NYC: compared to proper 
projected coordinates (e.g. UTM) or Haversine distances, the error is < 0.1% at this scale.

In [9]:
# Note: parse_timestamps_2014 created a copy of df_2014. After this point,
# the original loaded DataFrame has been replaced and can be garbage collected.
df_2014, scaler = normalize_coordinates(df_2014)
print(f"lat_norm range: [{df_2014['lat_norm'].min():.4f}, {df_2014['lat_norm'].max():.4f}]")
print(f"lon_norm range: [{df_2014['lon_norm'].min():.4f}, {df_2014['lon_norm'].max():.4f}]")

lat_norm range: [0.0000, 1.0000]
lon_norm range: [0.0000, 1.0000]


In [10]:
zone_centroids = compute_zone_centroids(zones_gdf)
print(f"Zone centroids: {zone_centroids.shape}")
zone_centroids.head()

Zone centroids: (260, 5)


,LocationID,zone,borough,centroid_lat,centroid_lon
0,1,Newark Airport,EWR,40.691830,-74.174002
1,2,Jamaica Bay,Queens,40.616746,-73.831300
2,3,Allerton/Pelham Gardens,Bronx,40.864474,-73.847422
3,4,Alphabet City,Manhattan,40.723752,-73.976968
4,5,Arden Heights,Staten Island,40.552659,-74.188485


### 2.5 Preprocessing Summary

In [11]:
print("=" * 60)
print("PREPROCESSING SUMMARY")
print("=" * 60)
print(f"\n2014 Dataset:")
print(f"  Records: {len(df_2014):,}")
print(f"  Period: {df_2014['datetime'].min().date()} to {df_2014['datetime'].max().date()}")
print(f"  Columns: {df_2014.columns.tolist()}")
print(f"  Zone mapping rate: {df_2014['LocationID'].notna().mean()*100:.1f}%")
print(f"  Memory: {df_2014.memory_usage(deep=True).sum() / 1e6:.1f} MB")

print(f"\n2015 Dataset:")
print(f"  Records: {len(df_2015):,}")
print(f"  Period: {df_2015['datetime'].min().date()} to {df_2015['datetime'].max().date()}")
print(f"  Columns: {df_2015.columns.tolist()}")
print(f"  Zone mapping rate: {df_2015['zone'].notna().mean()*100:.1f}%")
print(f"  Memory: {df_2015.memory_usage(deep=True).sum() / 1e6:.1f} MB")

PREPROCESSING SUMMARY

2014 Dataset:
  Records: 4,503,753
  Period: 2014-04-01 to 2014-09-30
  Columns: ['Date/Time', 'Lat', 'Lon', 'Base', 'datetime', 'hour', 'day_of_week', 'month', 'is_weekday', 'year', 'LocationID', 'zone', 'borough', 'lat_norm', 'lon_norm']
  Zone mapping rate: 98.7%
  Memory: 1163.5 MB

2015 Dataset:
  Records: 14,264,215
  Period: 2015-01-01 to 2015-06-30
  Columns: ['Dispatching_base_num', 'Pickup_date', 'Affiliated_base_num', 'locationID', 'datetime', 'hour', 'day_of_week', 'month', 'is_weekday', 'year', 'LocationID', 'borough', 'zone']
  Zone mapping rate: 100.0%
  Memory: 3317.0 MB


## 3. Exploratory Data Analysis

### 3.1 Geographic Distribution (2014)

A density heatmap reveals the spatial concentration of pickups before any clustering. 
This motivates the clustering approach: we can visually confirm that pickups form dense 
zones rather than being uniformly distributed.

In [12]:
sample_map = df_2014.sample(n=min(20_000, len(df_2014)), random_state=42)

fig = px.density_map(
    sample_map,
    lat="Lat",
    lon="Lon",
    radius=5,
    zoom=10,
    center={"lat": 40.73, "lon": -73.98},
    map_style="open-street-map",
    title="Pickup Density — 2014 (20k sample)",
    height=600,
)
fig.write_image("reports/figures/02_01_density_map.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![02_01_density_map](reports/figures/02_01_density_map.png)

The density map confirms that pickups concentrate heavily in Manhattan, with visible activity extending into northwest Brooklyn. Scattered pickups appear across Queens (potentially near airports) and a small cluster is visible near Newark Airport (EWR). This non-uniform spatial distribution, dominated by Manhattan, is well suited for clustering algorithms.

### 3.2 Temporal Distributions

In [13]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Pickups by Hour", "Pickups by Day of Week"))

# Hourly
hourly = df_2014.groupby("hour").size().reset_index(name="count")
fig.add_trace(
    go.Bar(x=hourly["hour"], y=hourly["count"], name="2014", marker_color="#636EFA"),
    row=1, col=1,
)

# Daily
daily = df_2014.groupby("day_of_week").size().reset_index(name="count")
daily["day_name"] = daily["day_of_week"].map(lambda x: DAY_NAMES[x])
fig.add_trace(
    go.Bar(x=daily["day_name"], y=daily["count"], name="2014", marker_color="#636EFA"),
    row=1, col=2,
)

fig.update_layout(height=400, title_text="Temporal Patterns — 2014", showlegend=False)
fig.write_image("reports/figures/02_02_hourly_daily_bars.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![02_02_hourly_daily_bars](reports/figures/02_02_hourly_daily_bars.png)

**Observations:**
- **Hourly:** Pickups build steadily from a 3-4 AM trough, reach a plateau around 7-8 AM, then climb to the daily maximum at 5-6 PM (~335k). Volume stays elevated through the evening before declining after 10 PM, consistent with commuter and evening-outing demand.
- **Daily:** Thursday and Friday are the busiest days, followed by Wednesday. Saturday outperforms Monday and rivals Tuesday, suggesting significant weekend evening demand alongside the dominant weekday commuter pattern. Sunday is the quietest day.

### 3.3 Monthly Volume Trend

In [14]:
monthly_2014 = df_2014.groupby("month").size().reset_index(name="count")
monthly_2014["year"] = "2014"

monthly_2015 = df_2015.groupby("month").size().reset_index(name="count")
monthly_2015["year"] = "2015"

monthly = pd.concat([monthly_2014, monthly_2015])

fig = px.bar(
    monthly,
    x="month",
    y="count",
    color="year",
    barmode="group",
    title="Monthly Pickup Volume — 2014 vs 2015",
    labels={"month": "Month", "count": "Pickups", "year": "Year"},
    color_discrete_map={"2014": "#636EFA", "2015": "#EF553B"},
    height=400,
)
fig.update_xaxes(dtick=1)
fig.write_image("reports/figures/02_03_monthly_comparison.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![02_03_monthly_comparison](reports/figures/02_03_monthly_comparison.png)

**Observation:** 2015 shows approximately 3.5-4x higher volume than 2014 for the overlapping months (April-June), reflecting Uber's rapid growth in NYC. Both years also show intra-year upward trends: 2014 nearly doubles from April (~550k) to September (~1.05M), while 2015 grows from ~1.95M in January to ~2.8M in June. This growth factor must be accounted for in any year-over-year comparison — we should compare relative distributions (proportions), not absolute counts.

### 3.4 Borough Distribution

In [15]:
boro_2014 = df_2014.groupby("borough").size().reset_index(name="count")
boro_2014["year"] = "2014"
boro_2014["pct"] = boro_2014["count"] / boro_2014["count"].sum() * 100

boro_2015 = df_2015.groupby("borough").size().reset_index(name="count")
boro_2015["year"] = "2015"
boro_2015["pct"] = boro_2015["count"] / boro_2015["count"].sum() * 100

boro = pd.concat([boro_2014, boro_2015])

fig = px.bar(
    boro,
    x="borough",
    y="pct",
    color="year",
    barmode="group",
    title="Pickup Distribution by Borough — 2014 vs 2015 (%)",
    labels={"borough": "Borough", "pct": "% of Total", "year": "Year"},
    color_discrete_map={"2014": "#636EFA", "2015": "#EF553B"},
    height=400,
)
fig.write_image("reports/figures/02_04_borough_distribution.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![02_04_borough_distribution](reports/figures/02_04_borough_distribution.png)

**Observation:** Manhattan dominates both years but its share dropped from ~77% (2014) to ~72% (2015), while Brooklyn (+3pp to ~16%) and Queens (+1.5pp to ~9.5%) gained share. This confirms Uber's expansion into outer boroughs between the two periods. Bronx, Staten Island, and EWR remain negligible (<2% each).

### 3.5 Dispatch Base Analysis (2014)

In [16]:
base_counts = df_2014.groupby("Base").size().reset_index(name="count")
base_counts = base_counts.sort_values("count", ascending=False)

fig = px.bar(
    base_counts,
    x="Base",
    y="count",
    title="Pickups by Dispatch Base — 2014",
    labels={"Base": "Dispatch Base Code", "count": "Pickups"},
    height=400,
)
fig.write_image("reports/figures/02_05_dispatch_bases.png", width=1200, height=700, scale=2)
fig.show()

### Expected output

![02_05_dispatch_bases](reports/figures/02_05_dispatch_bases.png)

**Observation:** Only 5 dispatch bases operate in the 2014 data. The top 3 (B02617, B02598, B02682) handle ~89% of all pickups with relatively similar volumes (1.2-1.4M each), while the remaining two (B02764, B02512) account for only ~10% combined (~200-260k each). This two-tier concentration suggests that Uber's NYC operations relied on a small core of high-volume dispatch bases.

## 4. Data Export

We save preprocessed data in Parquet format for efficient loading in subsequent notebooks:
- **NB03 (Clustering):** needs 2014 data with normalized coordinates
- **NB04 (Hot-Zone Analysis):** needs both 2014 and 2015 data with zone mappings

In [17]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_2014.to_parquet(OUTPUT_DIR / "preprocessed_2014.parquet", index=False)
print(f"Saved 2014: {len(df_2014):,} rows -> {OUTPUT_DIR / 'preprocessed_2014.parquet'}")

df_2015.to_parquet(OUTPUT_DIR / "preprocessed_2015.parquet", index=False)
print(f"Saved 2015: {len(df_2015):,} rows -> {OUTPUT_DIR / 'preprocessed_2015.parquet'}")

zone_centroids.to_csv(OUTPUT_DIR / "zone_centroids.csv", index=False)
print(f"Saved zone centroids: {len(zone_centroids)} zones -> {OUTPUT_DIR / 'zone_centroids.csv'}")

print("\nData ready for NB03 (Clustering) and NB04 (Hot-Zone Analysis).")

Saved 2014: 4,503,753 rows -> /home/sambot/dsfs/000_PROJECTS/UBER/data/output/preprocessed_2014.parquet
Saved 2015: 14,264,215 rows -> /home/sambot/dsfs/000_PROJECTS/UBER/data/output/preprocessed_2015.parquet
Saved zone centroids: 260 zones -> /home/sambot/dsfs/000_PROJECTS/UBER/data/output/zone_centroids.csv

Data ready for NB03 (Clustering) and NB04 (Hot-Zone Analysis).


**Note:** The MinMaxScaler is fitted on `[Lat, Lon × cos(40.7128°)]`, not raw `[Lat, Lon]`. 
It is not exported because NB03 will refit it on the same data — both notebooks import 
`normalize_coordinates` from `preprocessing.py`, so the same cosine correction is applied 
consistently. The normalized columns `lat_norm` and `lon_norm` are already saved in the 
Parquet file. To recover real longitude from the scaler's inverse transform, divide the 
`Lon_corr` output by `cos(radians(40.7128))` — the inverse returns `[Lat, Lon_corrected]`, 
not raw `[Lat, Lon]`.

## Next Steps

- **NB03 — Clustering Analysis:** Compare KMeans and DBScan on the 2014 GPS data, select optimal 
  parameters, and assign cluster labels.
- **NB04 — Hot-Zone Analysis:** Analyze hot-zones by time of day/week, and compare year-over-year 
  evolution at the zone level.